# ANSR-DT / HNSRM PoC - C-MAPSS FD001 walkthrough

This notebook walks through a complete run of the lightweight hybrid
neuro-symbolic pipeline: LightGBM perception, training-only symbolic
thresholds, ProbLog failure risk, validation-only probability fusion,
alert confirmation and the human-in-the-loop queue.

It has two modes:

1. **Fast mode (default)** - reads the artefacts already written to
   `results/` by the last full run, so every cell finishes in seconds and
   nothing is retrained.
2. **Full mode (optional, section 10)** - re-runs `main.py` end to end
   (about 35 minutes) to regenerate `results/` and the README tables.

Prerequisites: `pip install -r requirements.txt`, the FD001 processed
parquet under `data/raw/CMAPSS/`, and SWI-Prolog only if you intend to
re-run the pipeline with the Prolog layer enabled.

## Setup

The project root is discovered from the current directory so the
notebook runs both from `notebooks/` and from the repository root.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results'

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 50)

required = [
    'final_run_results.json',
    'calibration_report.json',
    'fusion_threshold.json',
    'temporal_thresholds.json',
    'hitl_queue.json',
]
missing = [name for name in required if not (RESULTS_DIR / name).exists()]

print(f'Project root : {PROJECT_ROOT}')
print(f'Results dir  : {RESULTS_DIR}')
print(f'Missing artefacts: {missing or "none"}')

## 1. Run under inspection

`final_run_results.json` is the run summary written by `main.py`: dataset,
split sizes, metrics, thresholds, fusion coefficients, alerting report and
the symbolic provenance record.

In [ ]:
summary = json.loads((RESULTS_DIR / 'final_run_results.json').read_text())
splits = summary['splits']

overview = pd.Series({
    'dataset': summary['dataset'],
    'architecture': summary['architecture'],
    'test source': summary['test_source'],
    'data file': Path(summary['data_file']).name,
    'run timestamp': summary.get('run_timestamp', ''),
    'train windows': splits['train_windows'],
    'validation windows': splits['validation_windows'],
    'test windows': splits['test_windows'],
    'train positive rate': round(splits['train_positive_rate'], 4),
    'test positive rate': round(splits['test_positive_rate'], 4),
})
overview.to_frame('value')

The symbolic facts in `rules.pl` are re-derived from the training engines
on every run and compared with the file. A mismatch aborts the run instead
of silently skewing the explanations with stale thresholds.

In [ ]:
provenance = summary['symbolic_rule_provenance']
print(f"checked facts  : {provenance['checked_facts']}")
print(f"split          : {provenance['split']} ({provenance['train_engines']} engines)")
print(f"mismatches     : {provenance['mismatches'] or 'none'}")
print(f"rules file     : {Path(provenance['rules_path']).name}")

## 2. Neural perception vs fused decision

The reported metrics are the fused ones. `neural_metrics` keeps the
bare LightGBM scores so the contribution of the symbolic signals stays
auditable.

In [ ]:
METRIC_KEYS = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision']
fusion = summary['probability_fusion']

rows = []
for split in ('validation', 'test'):
    sources = {
        'neural only': summary['neural_metrics'][split],
        'fused': fusion.get(f'{split}_metrics', {}),
        'reported': summary['metrics'][split],
    }
    for label, metrics in sources.items():
        rows.append({
            'split': split,
            'source': label,
            **{key: metrics.get(key) for key in METRIC_KEYS},
        })

metrics_table = pd.DataFrame(rows).set_index(['split', 'source'])
metrics_table.round(4)

In [ ]:
confusion = pd.DataFrame(
    [
        {
            'split': split,
            'true negatives': summary['metrics'][split]['confusion_matrix'][0][0],
            'false positives': summary['metrics'][split]['confusion_matrix'][0][1],
            'false negatives': summary['metrics'][split]['confusion_matrix'][1][0],
            'true positives': summary['metrics'][split]['confusion_matrix'][1][1],
        }
        for split in ('validation', 'test')
    ]
).set_index('split')
confusion

## 3. Decision threshold and precision/recall trade-off

The threshold is selected on validation only. With fusion active the
reported threshold is the fusion threshold; the neural threshold is kept
alongside it for reference.

In [ ]:
threshold = summary['decision_threshold']
print(f"source               : {threshold['source']}")
print(f"strategy             : {threshold['strategy']}")
print(f"active threshold     : {threshold['threshold']:.4f}")
print(f"neural-only threshold: {threshold['neural_threshold']:.4f}")
print(f"fusion trained on    : {fusion['trained_on']}")
print(f"fusion features      : {', '.join(fusion['features'])}")

pd.Series(fusion['coefficients'], name='coefficient').to_frame()

In [ ]:
curve = json.loads((RESULTS_DIR / 'fusion_threshold.json').read_text())
raw_curve = curve['precision_recall_curve']
pr = pd.DataFrame({
    'precision': raw_curve['precision'],
    'recall': raw_curve['recall'],
})

print(
    f"candidates evaluated: {curve['candidates_evaluated']}  "
    f"beta: {curve['beta']}  min precision: {curve['min_precision']}"
)

ax = pr.plot(
    x='recall', y='precision', figsize=(6, 4), legend=False, grid=True
)
ax.set_title('Fusion precision/recall on validation')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
plt.tight_layout()
plt.show()

## 4. Probability calibration

Sigmoid and isotonic calibrators are fitted on training data and compared
on validation; the test split never drives the choice.

In [ ]:
calibration = summary['calibration']
print(
    f"method: {calibration['method']} "
    f"(requested {calibration['requested_method']}, "
    f"selected on {calibration['selection_metric']})"
)
pd.DataFrame(calibration['candidates']).set_index('method').round(5)

In [ ]:
rows = []
for split in ('validation', 'test'):
    metrics = calibration['metrics'][split]
    rows.append({
        'split': split,
        'brier score': metrics['brier_score'],
        'ece': metrics['ece'],
        'mean predicted': metrics['mean_predicted_probability'],
        'observed positive rate': metrics['positive_rate'],
    })
pd.DataFrame(rows).set_index('split').round(4)

In [ ]:
report = json.loads((RESULTS_DIR / 'calibration_report.json').read_text())

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, split in zip(axes, ('validation', 'test')):
    curve_split = report[split]['reliability_curve']
    predicted = np.asarray(curve_split['mean_predicted'])
    observed = np.asarray(curve_split['fraction_positive'])
    counts = np.asarray(curve_split['counts'])
    populated = counts > 0
    ax.plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=1)
    ax.plot(predicted[populated], observed[populated], marker='o')
    ax.set_title(
        f"{split} - Brier {report[split]['brier_score']:.4f}, "
        f"ECE {report[split]['ece']:.4f}"
    )
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed positive rate')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Alert confirmation

A single noisy window should not raise an alert. The report compares
immediate alerts with `k` consecutive positives and a trailing moving
average of the risk score, on the held-out engines.

In [ ]:
alerting = summary['alerting']
rows = []
for name, mode in alerting['modes'].items():
    rows.append({
        'mode': name,
        'mean recall': mode['mean_recall'],
        'mean precision': mode['mean_precision'],
        'false alarms': mode['total_false_alarms'],
        'mean lead time': mode['mean_lead_time_cycles'],
        'alerted before EOL': mode['proportion_alerted_before_eol_among_relevant'],
    })
pd.DataFrame(rows).set_index('mode').round(4)

## 6. Engine-level behaviour

Window metrics hide per-engine behaviour, so the pipeline also exports
recall, precision, false alarms, first alert cycle and lead time for each
engine.

In [ ]:
engine_summary = pd.DataFrame({
    split: summary['engine_metrics'][split]['summary']
    for split in ('validation', 'test')
})
engine_summary.round(4)

In [ ]:
engine_files = {
    'validation': RESULTS_DIR / 'engine_metrics_validation.csv',
    'test': RESULTS_DIR / 'engine_metrics_test.csv',
    'validation (fusion)': RESULTS_DIR / 'engine_metrics_fusion_validation.csv',
    'test (fusion)': RESULTS_DIR / 'engine_metrics_fusion_test.csv',
}
engine_frames = {
    name: pd.read_csv(path) for name, path in engine_files.items() if path.exists()
}
print('Loaded:', ', '.join(engine_frames) or 'none')

engine_frame = engine_frames.get('test (fusion)', engine_frames.get('test'))
if engine_frame is None:
    print('No test engine metrics available.')
else:
    relevant = engine_frame[engine_frame['positive_windows'] > 0]
    lead = engine_frame['lead_time_cycles'].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(relevant['unit'].astype(str), relevant['false_alarms'])
    axes[0].set_title('False alarms per relevant engine (test)')
    axes[0].set_xlabel('engine')
    axes[0].set_ylabel('false alarms')
    axes[0].tick_params(axis='x', labelrotation=90, labelsize=7)

    axes[1].hist(lead, bins=10)
    axes[1].set_title('Lead time at first correct alert (test)')
    axes[1].set_xlabel('cycles before end of life')
    axes[1].set_ylabel('engines')

    plt.tight_layout()
    plt.show()
    print(
        f'Mean lead time: {lead.mean():.1f} cycles over '
        f'{len(lead)} alerted engines'
    )

## 7. Symbolic layer: thresholds and temporal facts

Every threshold in `rules.pl` is derived from the 20 training engines of
the 20/20/60 unit split. The same values feed SWI-Prolog reasoning and the
ProbLog model, so the symbolic layer cannot read the validation or test
engines. `python -m src.reasoning.thresholds --write` regenerates them.

In [ ]:
import importlib

thresholds_module = importlib.import_module('src.reasoning.thresholds')
rules_text = (PROJECT_ROOT / 'src' / 'reasoning' / 'rules.pl').read_text()
facts = thresholds_module.parse_threshold_facts(rules_text)

symbolic = pd.DataFrame({
    'high threshold': pd.Series(facts['sensor_high_threshold']),
    'low threshold': pd.Series(facts['sensor_low_threshold']),
    'gradient threshold': pd.Series(facts['gradient_threshold']),
}).sort_index()
print(f"{len(symbolic)} sensors with direction thresholds")
symbolic.round(3)

In [ ]:
named = pd.Series(facts['hpc_rule_threshold']).rename_axis(
    ['sensor', 'kind']
).to_frame('threshold')
named

In [ ]:
temporal = pd.DataFrame(
    json.loads((RESULTS_DIR / 'temporal_thresholds.json').read_text())
).T
temporal = temporal[[
    'slope_abs', 'relative_change_abs', 'std', 'persistence',
    'moving_average_cycles', 'training_windows',
]]
temporal

## 8. ProbLog failure risk

The Prolog conditions are combined probabilistically. The three reference
states below are the ones written to `results/prob_log_examples.json`: a
healthy engine, a degraded one, and one with high temperature and low
coolant bleed.

In [ ]:
try:
    from src.reasoning.prob_query import EXAMPLES, RULES_PATH, evaluate_state
    problog_error = None
except ImportError as error:
    EXAMPLES, RULES_PATH, evaluate_state = {}, None, None
    problog_error = error

rows = []
if EXAMPLES:
    program = Path(RULES_PATH).read_text()
    for label, sensor_values in EXAMPLES.items():
        result = evaluate_state(program, sensor_values)
        rows.append({
            'state': label,
            'P(failure_risk)': round(result['failure_risk'], 4),
            'P(hpc_degradation)': round(result['hpc_degradation'], 4),
            'active conditions': ', '.join(result['active_conditions']) or '-',
        })
if problog_error is not None:
    print(f'ProbLog is not installed in this environment: {problog_error}')

pd.DataFrame(rows).set_index('state') if rows else 'No ProbLog table available.'

## 9. Human-in-the-loop queue

Windows where the neural probability and the symbolic evidence disagree,
or where the probability sits close to the decision boundary, are exported
for review. Accepted labels become rules in `rules_learned.pl` and are
validated on the validation split before being used.

In [ ]:
queue = json.loads((RESULTS_DIR / 'hitl_queue.json').read_text())
meta = queue['metadata']
print(
    f"decision threshold {meta['decision_threshold']:.4f}, "
    f"uncertainty band [{meta['uncertain_low']:.4f}, {meta['uncertain_high']:.4f}]"
)
print(f"cases exported: {len(queue['cases'])} (limit {meta['max_cases']})")

cases = pd.DataFrame(queue['cases'])
preview = cases[[
    'case_id', 'unit', 'cycle', 'neural_probability',
    'problog_failure_risk', 'reason', 'insights',
]].head(10)
preview['insights'] = preview['insights'].apply(
    lambda items: ', '.join(items) if isinstance(items, list) else ''
)
preview.round(4)

In [ ]:
hitl_run = summary['hitl_queue']
print(f"candidates evaluated: {hitl_run['candidates_evaluated']}")
print(f"eligible records    : {hitl_run['eligible_records']}")
print(f"queue reasons       : {hitl_run['reasons']}")
print(f"learned-rule check  : {summary['hitl_rule_validation']['summary']}")

## 10. Optional: full pipeline run

Set `RUN_FULL_PIPELINE = True` to retrain everything and regenerate
`results/`. This takes roughly 35 minutes on a laptop and needs
SWI-Prolog for the symbolic layer.

Because the provenance check is blocking, changing `train_fraction`,
`validation_split` or the source parquet requires recomputing the symbolic
facts first:

```bash
python -m src.reasoning.thresholds --write
```

Only run the cell below once the thresholds match the training split.

In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    import subprocess

    command = [sys.executable, 'main.py']
    print('Running:', ' '.join(command))
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, text=True, capture_output=True
    )
    print(completed.stdout[-4000:])
    completed.check_returncode()
else:
    print(
        'Skipped. Set RUN_FULL_PIPELINE = True to retrain and '
        'regenerate results/.'
    )

## Notes

- Test numbers come from the held-out 60 % of FD001 engines; the
  validation split drives calibration, threshold selection and fusion.
- Window-level precision is low by construction (~3 % positive rate), so
  read it together with recall, ROC AUC and the engine-level tables.
- `tests/test_readme_metrics.py` pins the README tables to the run
  recorded in `results/`, so a stale notebook output is easy to spot.
- `python -m unittest discover -s tests` runs the full suite; the symbolic tests
  are skipped automatically when SWI-Prolog is missing.